# Final Incident Dataset Integration

In [1]:
# Import pandas so we can work with CSV files and DataFrames
import pandas as pd

/Users/nadiacamacho/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Step 1: Read in the CSV files

In [2]:
audio_df = pd.read_csv("audio_output.csv")
pdf_df = pd.read_csv("pdf.csv")
image_df = pd.read_csv("Image.csv")
text_df = pd.read_csv("Text.csv")

In [3]:
# Looking at what is stored in each DataFrame.
print("Audio columns:", audio_df.columns.tolist())
print("PDF columns:", pdf_df.columns.tolist())
print("Image columns:", image_df.columns.tolist())
print("Text columns:", text_df.columns.tolist())

Audio columns: ['Call_ID', 'Transcript', 'Extracted_Event', 'Location', 'Sentiment', 'Urgency_Score']
PDF columns: ['Report_ID', 'Incident_Type', 'Date', 'Location', 'Agency', 'Officer', 'Summary', 'Suspect_Description', 'Outcome', 'Source_Pages']
Image columns: ['Image_ID', 'Scene_Type', 'Objects_Detected', 'Text_Extracted', 'Fire_Detection_Confidence', 'Smoke_Detection_Confidence', 'Fire_Classification_Confidence', 'Scene_Decision_Confidence']
Text columns: ['Text_ID', 'Source', 'Raw_Text', 'Sentiment', 'Entities', 'Topic']


In [4]:
# Checking the number of rows and columns in each file.
print("Audio shape:", audio_df.shape)
print("PDF shape:", pdf_df.shape)
print("Image shape:", image_df.shape)
print("Text shape:", text_df.shape)

Audio shape: (20, 6)
PDF shape: (27, 10)
Image shape: (2006, 8)
Text shape: (115, 6)


## Step 2: Put each file into the same final format

The files have different column names because each modality had a different task. Before we can union/concatenate them, we need each one to have the same final columns.


In [5]:
# Creating the final audio DataFrame.
# We keep the audio Call_ID, but we also make sure it uses the AUD prefix. The transcript does not have a real time column, so Time is set to Unknown.
audio_final = pd.DataFrame()
audio_final["Incident_ID"] = [f"AUD-{i+1:03d}" for i in range(len(audio_df))]
audio_final["Original_ID"] = audio_df["Call_ID"]
audio_final["Source"] = "Audio"
audio_final["Event"] = audio_df["Extracted_Event"]
audio_final["Location"] = audio_df["Location"]
audio_final["Time"] = "Unknown"
audio_final["Score"] = audio_df["Urgency_Score"] * 10

audio_final.head()

,Incident_ID,Original_ID,Source,Event,Location,Time,Score
0,AUD-001,AUD-001,Audio,Homicide,Unknown,Unknown,6.7
1,AUD-002,AUD-002,Audio,Homicide,Unknown,Unknown,6.7
2,AUD-003,AUD-003,Audio,Unknown,Unknown,Unknown,0.0
3,AUD-004,AUD-004,Audio,Homicide,Unknown,Unknown,3.3
4,AUD-005,AUD-005,Audio,Unknown,Unknown,Unknown,0.0


In [6]:
# Creating the final PDF DataFrame.
# Since most PDF records are administrative or policy documents rather than direct incident reports, each record is assigned a default score of 5.

pdf_final = pd.DataFrame()

pdf_final["Incident_ID"] = [f"DOC-{i+1:03d}" for i in range(len(pdf_df))]
pdf_final["Original_ID"] = pdf_df["Report_ID"]
pdf_final["Source"] = "PDF"
pdf_final["Event"] = pdf_df["Incident_Type"]
pdf_final["Location"] = pdf_df["Location"]
pdf_final["Time"] = pdf_df["Date"]

pdf_final["Score"] = 5

pdf_final.head()

,Incident_ID,Original_ID,Source,Event,Location,Time,Score
0,DOC-001,RPT_001,PDF,Vendor Capability Statement,"Bentonville, Arkansas 72712","May 26, 2015",5
1,DOC-002,RPT_002,PDF,Policies and Procedures,NaN,6/1/15,5
2,DOC-003,RPT_003,PDF,Training Lesson Plan,NaN,4/24/2015,5
3,DOC-004,RPT_004,PDF,Policy/Correspondence,"Mike BEEBE East Camden, Arkansas 71711","July 1, 2014",5
4,DOC-005,RPT_005,PDF,Policy/Correspondence,"CABOT, AR",NaN,5


In [7]:
# Creating the final image DataFrame.
# The image file has several confidence columns so we'll use Scene_Decision_Confidence as the main confidence signal 
image_final = pd.DataFrame()
image_final["Incident_ID"] = [f"IMG-{i+1:03d}" for i in range(len(image_df))]
image_final["Original_ID"] = image_df["Image_ID"]
image_final["Source"] = "Image"
image_final["Event"] = image_df["Scene_Type"]
image_final["Location"] = "Unknown"
image_final["Time"] = "Unknown"
image_final["Score"] = image_df["Scene_Decision_Confidence"] * 10

image_final.head()

,Incident_ID,Original_ID,Source,Event,Location,Time,Score
0,IMG-001,0341cc4e-73c8-4f12-87c6-732f4d94b199_jpg.rf.7e...,Image,Forest Fire,Unknown,Unknown,9.11
1,IMG-002,056bfb4e-331b-4a89-b282-3bc809acb9bb_jpg.rf.d4...,Image,Forest Fire,Unknown,Unknown,9.76
2,IMG-003,0ce61215-6d34-49f9-aa6d-5f0feed430f8_jpg.rf.d7...,Image,Grass Fire,Unknown,Unknown,10.00
3,IMG-004,0d129596-761e-40bd-b132-ada4ade34a82_jpg.rf.15...,Image,Structure Fire,Unknown,Unknown,10.00
4,IMG-005,0d1ad876-45a6-46c0-aaaa-c33c4324ba82_jpg.rf.dd...,Image,Structure Fire,Unknown,Unknown,10.00


In [8]:
# The text output already classifies each post into an incident topic, so we used topic as the event field and because the dataset doesn't contain 
# separate location or time information, those fields are set to Unknown.

text_final = pd.DataFrame()

text_final["Incident_ID"] = [f"TXT-{i+1:03d}" for i in range(len(text_df))]
text_final["Original_ID"] = text_df["Text_ID"]
text_final["Source"] = "Text"
text_final["Event"] = text_df["Topic"]
text_final["Location"] = "Unknown"
text_final["Time"] = "Unknown"

# We manually assigned scores based on the type of incident. More serious incidents such as violence or arson receive higher scores.

topic_score_map = {
    "Assault / Violence": 9,
    "Fire / Arson": 8,
    "Theft / Robbery": 7,
    "Traffic Accident": 6,
    "Public Disturbance": 4,
    "Other": 5
}

# Assign a score to each record based on its topic.
text_final["Score"] = text_df["Topic"].map(topic_score_map)

# If a topic is not found in the dictionary above,
# assign it a default score of 5.
text_final["Score"] = text_final["Score"].fillna(5)

text_final.head()

,Incident_ID,Original_ID,Source,Event,Location,Time,Score
0,TXT-001,TXT_1,Text,Assault / Violence,Unknown,Unknown,9
1,TXT-002,TXT_2,Text,Assault / Violence,Unknown,Unknown,9
2,TXT-003,TXT_3,Text,Public Disturbance,Unknown,Unknown,4
3,TXT-004,TXT_4,Text,Public Disturbance,Unknown,Unknown,4
4,TXT-005,TXT_5,Text,Other,Unknown,Unknown,5


## Step 3: Combine the files together

This step uses `pd.concat`, which stacks the rows on top of each other.  
We are **not joining** the files because the project instructions say the modalities do not share the same incidents.


In [9]:
# This step merges/combines the files by stacking the rows together.

final_df = pd.concat(
    [audio_final, pdf_final, image_final, text_final],
    ignore_index=True
)
final_df.head()

,Incident_ID,Original_ID,Source,Event,Location,Time,Score
0,AUD-001,AUD-001,Audio,Homicide,Unknown,Unknown,6.7
1,AUD-002,AUD-002,Audio,Homicide,Unknown,Unknown,6.7
2,AUD-003,AUD-003,Audio,Unknown,Unknown,Unknown,0.0
3,AUD-004,AUD-004,Audio,Homicide,Unknown,Unknown,3.3
4,AUD-005,AUD-005,Audio,Unknown,Unknown,Unknown,0.0


In [10]:
# Checking the final number of rows.
final_df.shape

(2168, 7)

## Step 4: Fill missing values

In [11]:
# Filling missing values with N/A. This is required because not every source has the same kind of information.

final_df = final_df.fillna("N/A")

# Looking at the first few rows after filling missing values.
final_df.head()

,Incident_ID,Original_ID,Source,Event,Location,Time,Score
0,AUD-001,AUD-001,Audio,Homicide,Unknown,Unknown,6.7
1,AUD-002,AUD-002,Audio,Homicide,Unknown,Unknown,6.7
2,AUD-003,AUD-003,Audio,Unknown,Unknown,Unknown,0.0
3,AUD-004,AUD-004,Audio,Homicide,Unknown,Unknown,3.3
4,AUD-005,AUD-005,Audio,Unknown,Unknown,Unknown,0.0


## Step 5: Create the final severity column
- 0 to less than 3 = Low
- 3 to less than 7 = Medium
- 7 to 10 = High

In [12]:
# Creating Severity based on the Score column. This uses the score scale explained above.

final_df["Score"] = pd.to_numeric(final_df["Score"], errors="coerce").fillna(5)

final_df.loc[final_df["Score"] < 3, "Severity"] = "Low"
final_df.loc[(final_df["Score"] >= 3) & (final_df["Score"] < 7), "Severity"] = "Medium"
final_df.loc[final_df["Score"] >= 7, "Severity"] = "High"

final_df.head()

,Incident_ID,Original_ID,Source,Event,Location,Time,Score,Severity
0,AUD-001,AUD-001,Audio,Homicide,Unknown,Unknown,6.7,Medium
1,AUD-002,AUD-002,Audio,Homicide,Unknown,Unknown,6.7,Medium
2,AUD-003,AUD-003,Audio,Unknown,Unknown,Unknown,0.0,Low
3,AUD-004,AUD-004,Audio,Homicide,Unknown,Unknown,3.3,Medium
4,AUD-005,AUD-005,Audio,Unknown,Unknown,Unknown,0.0,Low


In [13]:
# Keeping the main required columns first. We also kept Original_ID and Score at the end because they are useful for checking our work.

final_df = final_df[["Incident_ID", "Source", "Event", "Location", "Time", "Severity", "Original_ID", "Score"]]

final_df.head()

,Incident_ID,Source,Event,Location,Time,Severity,Original_ID,Score
0,AUD-001,Audio,Homicide,Unknown,Unknown,Medium,AUD-001,6.7
1,AUD-002,Audio,Homicide,Unknown,Unknown,Medium,AUD-002,6.7
2,AUD-003,Audio,Unknown,Unknown,Unknown,Low,AUD-003,0.0
3,AUD-004,Audio,Homicide,Unknown,Unknown,Medium,AUD-004,3.3
4,AUD-005,Audio,Unknown,Unknown,Unknown,Low,AUD-005,0.0


In [14]:
# Checking how many incidents came from each source.

final_df["Source"].value_counts()

Source
Image    2006
Text      115
PDF        27
Audio      20
Name: count, dtype: int64

In [15]:
# Checking how many incidents are Low, Medium, and High.

final_df["Severity"].value_counts()

Severity
High      1926
Medium     132
Low        110
Name: count, dtype: int64

## Step 6: Save the final integrated dataset

In [16]:
# Saving the final integrated dataset as a CSV file.

final_df.to_csv("final_integrated_incidents.csv", index=False)